# Training and Evaluation in one Notebook for One Model-Database Pair

# To check before running
1. Check class names for your event log in the **p2pencoder.py** ( *{event_log_name}encoder.py* )
2. Check the Axioms in **axiombuilder.py**
3. make sure you have done the declare mining on the event log and have a valid **ltn_rows_path**

In [1]:
event_log_name = "large"
if event_log_name is None:
    raise ValueError("Please set the event_log_name variable to the name of the event log you want to use.")
ltn_rows_path = f"{event_log_name}_ltn_rows.pkl"
print(f"Event log name {event_log_name}")
print(f"LTN Rows path {ltn_rows_path}")
# starting time


Event log name large
LTN Rows path large_ltn_rows.pkl


In [2]:
# import tensorflow as tf
# physical_devices = tf.config.list_physical_devices('GPU')
# print(physical_devices)
# if len(physical_devices) > 0:
#     tf.config.experimental.set_memory_growth(physical_devices[0], True)
#     print("GPU found")
#     print("Memory growth set")
# else:
#     print("No GPU found")

In [ ]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

import itertools

from sklearn import metrics


from april.anomalydetection import BINet
from april.anomalydetection.utils import label_collapse
from april.database import Evaluation
from april.largeevaluator import Evaluator
from april.fs import get_model_files
from april.fs import PLOT_DIR

import matplotlib.pyplot as plt
import numpy as np
np.random.seed(0)

import pandas as pd
import seaborn as sns
from sqlalchemy.orm import Session
import scikit_posthocs as sp

from april.database import get_engine
from april.fs import PLOT_DIR
from april.utils import microsoft_colors, prettify_dataframe, cd_plot, get_cd
from april.enums import Base, Strategy, Heuristic

sns.set_style('white')
pd.set_option('display.max_rows', 50)
%config InlineBackend.figure_format = 'retina'
print(large_leaky_row_classes)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set
Creating Evaluation table
[<class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-10'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-25'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-50'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-100'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-150'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-200'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-250'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-300'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-350'>]


In [68]:
dataset = f"{event_log_name}-0.3-1"
out_dir = PLOT_DIR / f'{event_log_name}_evaluations_both_{arrow.now().format("YYYY-MM-DD-HH-mm-ss")}'
eval_file = out_dir / f'{event_log_name}_fraction_evaluations.pkl'
csv_file = out_dir / f'{event_log_name}_fraction_evaluations.csv'
excel_file = out_dir / f'{event_log_name}_fraction_evaluations.xlsx'
model_folder = r"D:\LTNcoder\.out\models"
db = r"D:\LTNcoder\.out\april.db"

# create out_dir if it does not exist
if not out_dir.exists():
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {out_dir}")
from april.utils import delete_all_files_in_folder, delete_evaluation_and_model_tables
delete_all_files_in_folder(model_folder)
delete_evaluation_and_model_tables(db)
start_time = arrow.now("Europe/Berlin")


Created directory: d:\LTNcoder\.out\plots\large_evaluations_both_2025-08-09-16-57-34
Deleted all rows from Evaluation and Model tables.


# Training

In [69]:
def fit_and_save(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
    if ad_kwargs is None:
        ad_kwargs = {}
    if fit_kwargs is None:
        fit_kwargs = {}

    # Save start time
    start_time = arrow.now()

    # Dataset
    dataset = Dataset(dataset_name)

    # AD
    ad = ad(**ad_kwargs)

    # Train and save
    ad.fit(dataset, **fit_kwargs)
    file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
    model_file = ad.save(file_name)

    # Save end time
    end_time = arrow.now()

    # Cache result
    Evaluator(model_file.str_path).cache_result()

    # Calculate training time in seconds
    training_time = (end_time - start_time).total_seconds()

    # Write to database
    engine = get_engine()
    session = Session(engine)

    session.add(Model(creation_date=end_time.datetime,
                      algorithm=ad.name,
                      training_duration=training_time,
                      file_name=model_file.file,
                      training_event_log_id=EventLog.get_id_by_name(dataset_name),
                      training_host=socket.gethostname(),
                      hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
    session.commit()
    session.close()

    if isinstance(ad, NNAnomalyDetector):
        from keras.backend import clear_session
        clear_session()
    pass

In [70]:
ads = [
        dict(ad=LargeDAE, fit_kwargs=dict(epochs=6, batch_size=100)),
    ] + \
    [
        dict(ad=LEAKY_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100)) for LEAKY_ROW_CLASS 
        in large_leaky_row_classes
    ] + \
    [
        dict(ad=LTN_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100, epochs_ltn=3))
        for LTN_ROW_CLASS in large_ltn_row_classes
    ]
print(ads)
for ad in tqdm(ads, desc="Fitting ADs"):
    fit_and_save(dataset, **ad)


[{'ad': <class 'april.anomalydetection.largeencoder.LargeDAE'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-10'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-25'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-50'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-100'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-150'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-200'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-250'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetectio

Fitting ADs:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 1/6
42/42 [==============================] - 1s 15ms/step - loss: 0.1768 - accuracy: 0.1871 - val_loss: 0.0152 - val_accuracy: 0.4642
Epoch 2/6
42/42 [==============================] - 0s 8ms/step - loss: 0.0067 - accuracy: 0.3787 - val_loss: 0.0051 - val_accuracy: 0.5249
Epoch 3/6
42/42 [==============================] - 0s 8ms/step - loss: 0.0054 - accuracy: 0.4103 - val_loss: 0.0049 - val_accuracy: 0.6876
Epoch 4/6
42/42 [==============================] - 0s 8ms/step - loss: 0.0052 - accuracy: 0.4593 - val_loss: 0.0048 - val_accuracy: 0.8937
Epoch 5/6
42/42 [==============================] - 0s 8ms/step - loss: 0.0051 - accuracy: 0.4852 - val_loss: 0.0044 - val_accuracy: 0.9479
Epoch 6/6
42/42 [==============================] - 0s 8ms/step - loss: 0.0047 - accuracy: 0.5083 - val_loss: 0.0039 - val_accuracy: 0.8178
d:\LTNcoder\.out\models\large-0.3-1_largedae_20250809-165734.368900.keras
Loading model large-0.3-1_largedae_20250809-165734.368900 / <april.fs.ModelFile object at 0

In [71]:
print(AD) #Evaluator dependso on AD

{'binetv0': <class 'april.anomalydetection.binet.binet.BINetv0'>, 'binetv1': <class 'april.anomalydetection.binet.binet.BINetv1'>, 'binetv2': <class 'april.anomalydetection.binet.binet.BINetv2'>, 'binetv3': <class 'april.anomalydetection.binet.binet.BINetv3'>, 'likelihood': <class 'april.anomalydetection.boehmer.BoehmerLikelihoodAnomalyDetector'>, 'dae': <class 'april.anomalydetection.autoencoder.DAE'>, 'daeltn': <class 'april.anomalydetection.autoencoder.DAELTN'>, 'daeltnfrozen': <class 'april.anomalydetection.autoencoder.DAELTNFROZEN'>, 'largedae': <class 'april.anomalydetection.largeencoder.LargeDAE'>, 'largedae-leaky-10': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-10'>, 'largedae-leaky-100': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-100'>, 'largedae-leaky-150': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-150'>, 'largedae-leaky-200': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-200'>, 'largedae-leaky-25': <class 'april.an

# Evaluation

In [72]:
heuristics = [h for h in Heuristic.keys() if h not in [Heuristic.DEFAULT, Heuristic.MANUAL, Heuristic.RATIO,
                                                       Heuristic.MEDIAN, Heuristic.MEAN]]
params = [(Base.SCORES, Heuristic.DEFAULT, Strategy.SINGLE), *itertools.product([Base.SCORES], heuristics, Strategy.keys())]

In [73]:
def _evaluate(params):
    # print(f"Evaluating {params}...")
    e, base, heuristic, strategy = params

    session = Session(get_engine())
    model = session.query(Model).filter_by(file_name=e.model_file.name).first()
    session.close()
    # print(f"Model {model} loaded from Session.")

    if Model is None:
        print(f"Models from session: {model}")
    # Generate evaluation frames
    y_pred = e.binarizer.binarize(base=base, heuristic=heuristic, strategy=strategy, go_backwards=False)
    y_true = e.binarizer.get_targets()

    evaluations = []
    for axis in [0, 1, 2]:
        # print(f"Evaluating axis {axis}...")
        for i, attribute_name in enumerate(e.dataset.attribute_keys):
            # print(f"Evaluating attribute {attribute_name}...")
            def get_evaluation(label, precision, recall, f1):
                return Evaluation(model_id=model.id, file_name=model.file_name,
                                  label=label, perspective=perspective, attribute_name=attribute_name,
                                  axis=axis, base=base, heuristic=heuristic, strategy=strategy,
                                  precision=precision, recall=recall, f1=f1)

            perspective = 'Control Flow' if i == 0 else 'Data'
            if i > 0 and not e.ad_.supports_attributes:
                # print(f"Skipping attribute {attribute_name} for model {model} as it does not support attributes.")
                evaluations.append(get_evaluation('Normal', 0.0, 0.0, 0.0))
                evaluations.append(get_evaluation('Anomaly', 0.0, 0.0, 0.0))
            else:
                # print(f"Evaluating attribute {attribute_name} for model {model}...")
                yp = label_collapse(y_pred[:, :, i:i + 1], axis=axis).compressed()
                yt = label_collapse(y_true[:, :, i:i + 1], axis=axis).compressed()
                p, r, f, _ = metrics.precision_recall_fscore_support(yt, yp, labels=[0, 1])
                evaluations.append(get_evaluation('Normal', p[0], r[0], f[0]))
                evaluations.append(get_evaluation('Anomaly', p[1], r[1], f[1]))

    return evaluations

def evaluate(model_name):
    print(f"Evaluating {model_name}...")
    e = Evaluator(model_name)
    print(f"{e} loaded.")
    # print attributes of e
    print(f"e.model_file: {e.model_file}")
    print(f"e.model_name: {e.model_name}")
    print(f"e.eventlog_name: {e.eventlog_name}")
    print(f"e.dataset: {e.dataset}")
    print(f"e.result: {e.result}")
    
    
    _params = []
    for base, heuristic, strategy in params:
        if e.dataset.num_attributes == 1 and strategy in [Strategy.ATTRIBUTE, Strategy.POSITION_ATTRIBUTE]:
            continue
        if isinstance(e.ad_, BINet) and e.ad_.version == 0:
            continue
        if heuristic is not None and heuristic not in e.ad_.supported_heuristics:
            continue
        if strategy is not None and strategy not in e.ad_.supported_strategies:
            continue
        if base is not None and base not in e.ad_.supported_bases:
            continue
        # print(f"Adding parameters: {e}, {base}, {heuristic}, {strategy}")
        _params.append([e, base, heuristic, strategy])
    
    print(f"{_params} parameters to evaluate.")

    return [_e for p in _params for _e in _evaluate(p)]

In [74]:
models = sorted([m.name for m in get_model_files() if m.p == 0.3])# and 'real' in m.name])
print(f"Available Models: {models}")
evaluations = []
for model in tqdm(models, desc='Evaluate'):
    e = evaluate(model)
    evaluations.append(e)
# Write to database
session = Session(get_engine())
for e in evaluations:
    session.bulk_save_objects(e)
    session.commit()
session.close()

Available Models: ['large-0.3-1_largedae-leaky-100_20250809-165748.600110', 'large-0.3-1_largedae-leaky-10_20250809-165738.203350', 'large-0.3-1_largedae-leaky-150_20250809-165752.372462', 'large-0.3-1_largedae-leaky-200_20250809-165755.893325', 'large-0.3-1_largedae-leaky-250_20250809-165759.409164', 'large-0.3-1_largedae-leaky-25_20250809-165741.763954', 'large-0.3-1_largedae-leaky-300_20250809-165802.877871', 'large-0.3-1_largedae-leaky-350_20250809-165806.505947', 'large-0.3-1_largedae-leaky-50_20250809-165745.175556', 'large-0.3-1_largedae_20250809-165734.368900', 'large-0.3-1_largeltnfrozen-100_20250809-165913.540035', 'large-0.3-1_largeltnfrozen-10_20250809-165810.116776', 'large-0.3-1_largeltnfrozen-150_20250809-165934.977362', 'large-0.3-1_largeltnfrozen-200_20250809-170008.153597', 'large-0.3-1_largeltnfrozen-250_20250809-170041.763132', 'large-0.3-1_largeltnfrozen-25_20250809-165831.800409', 'large-0.3-1_largeltnfrozen-300_20250809-170112.508136', 'large-0.3-1_largeltnfrozen

Evaluate:   0%|          | 0/19 [00:00<?, ?it/s]

Evaluating large-0.3-1_largedae-leaky-100_20250809-165748.600110...
Loading model large-0.3-1_largedae-leaky-100_20250809-165748.600110 / <april.fs.ModelFile object at 0x000002422E2A4940> for event log large-0.3-1 at path d:\LTNcoder\.out\models\large-0.3-1_largedae-leaky-100_20250809-165748.600110.keras
Self.ad_: <april.anomalydetection.largeencoder.LargeDAE-Leaky-100 object at 0x000002422E2A4850>
<april.largeevaluator.Evaluator object at 0x000002422E2A46A0> loaded.
e.model_file: d:\LTNcoder\.out\models\large-0.3-1_largedae-leaky-100_20250809-165748.600110.keras
e.model_name: large-0.3-1_largedae-leaky-100_20250809-165748.600110
e.eventlog_name: large-0.3-1
Filtering dataset to 396 LTN rows.
Indices: [26, 29, 38, 49, 50, 65, 73, 74, 80, 81, 82, 88, 92, 112, 113, 115, 129, 179, 188, 193, 210, 220, 234, 239, 246, 249, 250, 251, 289, 339, 370, 375, 387, 391, 420, 441, 449, 451, 454, 463, 468, 473, 489, 492, 497, 499, 503, 510, 512, 521, 529, 541, 545, 547, 549, 551, 556, 571, 584, 589, 5

In [75]:

session = Session(get_engine())
evaluations = session.query(Evaluation).all()
rows = []

for ev in tqdm(evaluations):
    # print(f"Evaluation: {ev}")
    m = ev.model
    # print(f"Model: {m}")
    el = ev.model.training_event_log
    # print(f"Event log: {el}")
    rows.append([m.file_name, m.creation_date, m.hyperparameters, m.training_duration, m.training_host, m.algorithm, 
                 el.name, el.base_name, el.percent_anomalies, el.number,
                 ev.axis, ev.base, ev.heuristic, ev.strategy, ev.label, ev.attribute_name, ev.perspective, ev.precision, ev.recall, ev.f1])
session.close()
columns = ['file_name', 'date', 'hyperparameters', 'training_duration', 'training_host', 'ad',
           'dataset_name', 'process_model', 'noise', 'dataset_id',
           'axis', 'base', 'heuristic', 'strategy', 'label', 'attribute_name', 'perspective', 'precision', 'recall', 'f1']
evaluation = pd.DataFrame(rows, columns=columns)

evaluation.to_pickle(eval_file)

  0%|          | 0/5472 [00:00<?, ?it/s]

In [76]:
synth_datasets = ['paper', 'p2p', 'small', 'medium', 'large', 'huge', 'gigantic', 'wide']
bpic_datasets = ['bpic12', 'bpic13', 'bpic15', 'bpic17']
anonymous_datasets = ['real']
datasets = synth_datasets + bpic_datasets + anonymous_datasets
dataset_types = ['Synthetic', 'Real-life']

orig_ads = [ad['ad'].__name__ for ad in ads if "DAE" in ad['ad'].__name__]
new_ads = [ad['ad'].__name__ for ad in ads if "DAE" not in ad['ad'].__name__]
ads = orig_ads + new_ads

heuristics = [r'$best$', r'$default$', r'$elbow_\downarrow$', r'$elbow_\uparrow$', 
              r'$lp_\leftarrow$', r'$lp_\leftrightarrow$', r'$lp_\rightarrow$']
print(ads)

['LargeDAE', 'LargeDAE-Leaky-10', 'LargeDAE-Leaky-25', 'LargeDAE-Leaky-50', 'LargeDAE-Leaky-100', 'LargeDAE-Leaky-150', 'LargeDAE-Leaky-200', 'LargeDAE-Leaky-250', 'LargeDAE-Leaky-300', 'LargeDAE-Leaky-350', 'LargeLTNFROZEN-10', 'LargeLTNFROZEN-25', 'LargeLTNFROZEN-50', 'LargeLTNFROZEN-100', 'LargeLTNFROZEN-150', 'LargeLTNFROZEN-200', 'LargeLTNFROZEN-250', 'LargeLTNFROZEN-300', 'LargeLTNFROZEN-350']


In [77]:
evaluation = evaluation.query(f'ad in {ads} and label == "Anomaly"')

In [78]:
evaluation['perspective-label'] = evaluation['perspective'] + '-' + evaluation['label']
evaluation['attribute_name-label'] = evaluation['attribute_name'] + '-' + evaluation['label']
evaluation['dataset_type'] = 'Synthetic'
evaluation.loc[evaluation['process_model'].str.contains('bpic'), 'dataset_type'] = 'Real-life'
evaluation.loc[evaluation['process_model'].str.contains('real'), 'dataset_type'] = 'Real-life'

In [79]:
_filtered_evaluation = evaluation.query(f'ad in {ads} and (strategy == "{Strategy.ATTRIBUTE}"'
                                       f' or (strategy == "{Strategy.SINGLE}" and process_model == "bpic12")'
                                       f' or (strategy == "{Strategy.SINGLE}" and ad == "Naive+"))')

In [80]:
filtered_evaluation = _filtered_evaluation.query(f'heuristic == "{Heuristic.DEFAULT}"'
                                                 f' or (heuristic == "{Heuristic.LP_MEAN}" and ad in {orig_ads})'
                                                 f' or (heuristic == "{Heuristic.LP_LEFT}" and ad in {new_ads})'
                                                )

In [81]:
df = filtered_evaluation.query('axis == 0')
df = prettify_dataframe(df)
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name', 'perspective'])[['precision', 'recall', 'f1']].mean().reset_index()
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name'])[['precision', 'recall', 'f1']].mean().reset_index()
df['f1'] = 2 * df['recall'] * df['precision'] / (df['recall'] + df['precision'])

df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model', 'dataset_name'], values=['precision', 'recall', 'f1'])
df = df.fillna(0)
df = df.stack(1).stack(1).reset_index()
df.to_excel(str(out_dir / 'table.xlsx'), index=False)

# drop rows in column "axis" which have value "Attribute"
df = df.query('axis != "Attribute"')

# df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model'], values=['precision', 'recall', 'f1'], aggfunc=np.mean)

df.to_excel(str(excel_file), index=False)
df.to_csv(str(csv_file), index=False)
print(df)

    axis                  ad process_model dataset_name        f1  precision  \
0   Case            LargeDAE         Large  large-0.3-1  0.238168   0.577778   
1   Case   LargeDAE-Leaky-10         Large  large-0.3-1  0.278328   0.574468   
2   Case  LargeDAE-Leaky-100         Large  large-0.3-1  0.273014   0.597561   
3   Case  LargeDAE-Leaky-150         Large  large-0.3-1  0.338644   0.602041   
4   Case  LargeDAE-Leaky-200         Large  large-0.3-1  0.279958   0.578947   
5   Case   LargeDAE-Leaky-25         Large  large-0.3-1  0.266583   0.590909   
6   Case  LargeDAE-Leaky-250         Large  large-0.3-1  0.292251   0.590000   
7   Case  LargeDAE-Leaky-300         Large  large-0.3-1  0.356268   0.567568   
8   Case  LargeDAE-Leaky-350         Large  large-0.3-1  0.428240   0.585714   
9   Case   LargeDAE-Leaky-50         Large  large-0.3-1  0.332079   0.585714   
10  Case   LargeLTNFROZEN-10         Large  large-0.3-1  0.520010   0.636792   
11  Case  LargeLTNFROZEN-100         Lar

C:\Users\devas\AppData\Local\Temp\ipykernel_2028\661170537.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()
C:\Users\devas\AppData\Local\Temp\ipykernel_2028\661170537.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()


In [82]:
display(df)

,axis,ad,process_model,dataset_name,f1,precision,recall
0,Case,LargeDAE,Large,large-0.3-1,0.238168,0.577778,0.150000
1,Case,LargeDAE-Leaky-10,Large,large-0.3-1,0.278328,0.574468,0.183654
2,Case,LargeDAE-Leaky-100,Large,large-0.3-1,0.273014,0.597561,0.176923
3,Case,LargeDAE-Leaky-150,Large,large-0.3-1,0.338644,0.602041,0.235577
4,Case,LargeDAE-Leaky-200,Large,large-0.3-1,0.279958,0.578947,0.184615
5,Case,LargeDAE-Leaky-25,Large,large-0.3-1,0.266583,0.590909,0.172115
6,Case,LargeDAE-Leaky-250,Large,large-0.3-1,0.292251,0.590000,0.194231
7,Case,LargeDAE-Leaky-300,Large,large-0.3-1,0.356268,0.567568,0.259615
8,Case,LargeDAE-Leaky-350,Large,large-0.3-1,0.428240,0.585714,0.337500
9,Case,LargeDAE-Leaky-50,Large,large-0.3-1,0.332079,0.585714,0.231731


# End

In [83]:
end_time = arrow.now("Europe/Berlin")
print(f"Start time: {start_time}")
print(f"End time: {end_time}")
print(f"Duration: {end_time - start_time}")

Start time: 2025-08-09T16:57:34.309154+02:00
End time: 2025-08-09T17:03:18.977388+02:00
Duration: 0:05:44.668234
